In [ ]:
import torch
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
import numpy as np

model_path = "./final_model"
processor = SegformerImageProcessor.from_pretrained(model_path)
model = SegformerForSemanticSegmentation.from_pretrained(model_path)

model.to("cpu")
model.eval()

model.config.return_dict = False 

dummy_image = np.zeros((512, 512, 3), dtype=np.uint8)

inputs = processor(images=dummy_image, return_tensors="pt")
dummy_pixel_values = inputs["pixel_values"]

onnx_file_path = "segformer.onnx"

torch.onnx.export(
    model,
    (dummy_pixel_values,),
    onnx_file_path,
    export_params=True,
    opset_version=14,
    input_names=['pixel_values'],
    output_names=['logits'],
    dynamic_axes={
        'pixel_values': {0: 'batch_size', 2: 'height', 3: 'width'},
        'logits': {0: 'batch_size', 2: 'height', 3: 'width'}
    }
)

print(f"Model successfully exported to {onnx_file_path}")

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Model successfully exported to segformer.onnx
